
# ESTADÍSTICA: EJERCICIOS COMPLEMENTARIOS CODEX: SOLUCIONES

Estos ejercicios complementan el curso oficial de estadística. Están pensados para practicar los conceptos que se añadieron en los cuadernos enriquecidos: más precisión en descriptiva, inferencia y conceptos avanzados.

#### Nota Codex

Sigue el mismo método en todos los ejercicios: primero identifica el tipo de problema, luego elige la herramienta estadística y finalmente interpreta el resultado en lenguaje simple.



## Preparación

Ejecuta esta celda antes de empezar. Cargamos paquetes y el dataset de ventas usado en los ejercicios oficiales.


In [ ]:

# Te dejo creada la importación de paquetes.
# Simplemente ejecuta esta celda para que se carguen.
import math
import statistics

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

try:
    import statsmodels.api as sm
    from statsmodels.stats.outliers_influence import variance_inflation_factor
except ImportError:
    sm = None
    variance_inflation_factor = None

pd.set_option('display.max_columns', None)



Vamos a trabajar con el dataset de ventas `HistoricoVentas.csv`, igual que en el cuaderno oficial.


In [ ]:

df = pd.read_csv('../../00_Datasets/HistoricoVentas.csv', sep=',', index_col=0)
df.dropna(inplace=True)
df.head()


## Ejercicio 1: identificar tipos de variables


### Ejercicio a analizar

Observa las columnas de `df` y crea una tabla manual llamada `tipos_variables` con tres columnas: `variable`, `tipo` y `motivo`.

Clasifica al menos estas variables: `Product_Category`, `Warehouse`, `Product_Code` y `Order_Demand`.


In [ ]:

# Solución Codex
# Product_Category y Warehouse son cualitativas nominales.
# Product_Code parece un identificador/categoría, aunque tenga números en el código.
# Order_Demand es cuantitativa porque representa cantidad vendida.

tipos_variables = pd.DataFrame({
    'variable': ['Product_Category', 'Warehouse', 'Product_Code', 'Order_Demand'],
    'tipo': ['cualitativa nominal', 'cualitativa nominal', 'cualitativa nominal', 'cuantitativa discreta'],
    'motivo': [
        'categoría sin orden natural',
        'almacén sin orden natural',
        'identificador de producto, no cantidad numérica',
        'conteo o demanda de unidades'
    ]
})

tipos_variables



#### Nota Codex

No todo lo que parece numérico debe tratarse como cuantitativo. Un código de producto puede estar formado por números, pero estadísticamente funciona como una categoría.


## Ejercicio 2: resumen con describe


### Ejercicio a analizar

Usa `describe()` sobre la variable `Order_Demand`. Después responde: ¿qué te dicen la media, la mediana aproximada, el mínimo, el máximo y los cuartiles?


In [ ]:

# Solución Codex
resumen = df.Order_Demand.describe()
resumen



#### Nota Codex

`describe()` es una primera radiografía. Si la media está muy lejos de la mediana, suele haber asimetría o valores extremos. Si el máximo está muy alejado de Q3, conviene mirar outliers.


In [ ]:

# Solución alternativa Codex: extraer valores clave
print('Media:', df.Order_Demand.mean())
print('Mediana:', df.Order_Demand.median())
print('Q1:', df.Order_Demand.quantile(0.25))
print('Q3:', df.Order_Demand.quantile(0.75))
print('Mínimo:', df.Order_Demand.min())
print('Máximo:', df.Order_Demand.max())


## Ejercicio 3: cuartiles e IQR


### Ejercicio a analizar

Calcula Q1, Q2, Q3 y el IQR de `Order_Demand`.


In [ ]:

# Solución Codex
ventas = df.Order_Demand
q1 = ventas.quantile(0.25)
q2 = ventas.quantile(0.50)
q3 = ventas.quantile(0.75)
iqr = q3 - q1

print('Q1:', q1)
print('Q2 / mediana:', q2)
print('Q3:', q3)
print('IQR:', iqr)



#### Nota Codex

El IQR mide la dispersión del 50% central de los datos. Es más robusto que el rango total porque no depende directamente de los valores extremos.


## Ejercicio 4: outliers con la regla del IQR


### Ejercicio a analizar

Usando Q1, Q3 e IQR, calcula los límites inferior y superior para detectar outliers. Después cuenta cuántos valores de `Order_Demand` quedan fuera de esos límites.


In [ ]:

# Solución Codex
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = ventas[(ventas < limite_inferior) | (ventas > limite_superior)]

print('Límite inferior:', limite_inferior)
print('Límite superior:', limite_superior)
print('Número de outliers:', outliers.shape[0])
print('Porcentaje de outliers:', outliers.shape[0] / ventas.shape[0] * 100)



#### Nota Codex

Detectar outliers no significa borrarlos automáticamente. Primero hay que entender si son errores, casos raros pero reales, o información importante para el negocio.


## Ejercicio 5: media vs mediana con valores extremos


### Ejercicio a analizar

Compara la media y la mediana de `Order_Demand`. Luego calcula la media sin outliers según la regla IQR. ¿Qué cambia?


In [ ]:

# Solución Codex
ventas_sin_outliers = ventas[(ventas >= limite_inferior) & (ventas <= limite_superior)]

print('Media original:', ventas.mean())
print('Mediana original:', ventas.median())
print('Media sin outliers:', ventas_sin_outliers.mean())
print('Mediana sin outliers:', ventas_sin_outliers.median())



#### Nota Codex

La media suele moverse mucho cuando hay valores extremos. La mediana suele ser más estable. Por eso, en distribuciones muy asimétricas, la mediana puede describir mejor el centro típico.


## Ejercicio 6: asimetría y curtosis


### Ejercicio a analizar

Calcula la asimetría (`skew`) y la curtosis de `Order_Demand`. Interpreta si la distribución parece simétrica o no.


In [ ]:

# Solución Codex
asimetria = stats.skew(ventas)
curtosis = stats.kurtosis(ventas)

print('Asimetría:', asimetria)
print('Curtosis:', curtosis)



#### Nota Codex

Si la asimetría es positiva, la cola derecha suele ser más larga. Si es negativa, la cola izquierda pesa más. La curtosis alta suele indicar colas pesadas o valores extremos.


## Ejercicio 7: interpretar una correlación


### Ejercicio a analizar

Crea dos variables numéricas: `Order_Demand` y una versión transformada con logaritmo `log_demand`. Calcula la correlación de Pearson entre ambas e interpreta el resultado.


In [ ]:

# Solución Codex
log_demand = np.log1p(df.Order_Demand)
correlacion, pvalor = stats.pearsonr(df.Order_Demand, log_demand)

print('Correlación:', correlacion)
print('p-valor:', pvalor)



#### Nota Codex

Una correlación alta indica relación lineal fuerte, pero no causalidad. Aquí la relación existe porque `log_demand` se construye directamente a partir de `Order_Demand`.


## Ejercicio 8: chi-cuadrado con interpretación


### Ejercicio a analizar

Haz un test chi-cuadrado entre `Product_Category` y `Warehouse`. Imprime el p-valor e interpreta si hay asociación al 95% de confianza.


In [ ]:

# Solución Codex
tabla = pd.crosstab(df.Product_Category, df.Warehouse)
chi2, pvalor, gl, esperados = stats.chi2_contingency(tabla)

print('Chi-cuadrado:', chi2)
print('p-valor:', pvalor)
print('Grados de libertad:', gl)

if pvalor < 0.05:
    print('Rechazamos H0: hay asociación entre categoría y almacén.')
else:
    print('No rechazamos H0: no hay evidencia suficiente de asociación.')



#### Nota Codex

Chi-cuadrado trabaja con frecuencias de variables categóricas. La hipótesis nula dice que las variables son independientes.


## Ejercicio 9: intervalo de confianza con bootstrap


### Ejercicio a analizar

Calcula un intervalo de confianza bootstrap al 95% para la media de `Order_Demand`. Usa 1000 remuestreos.


In [ ]:

# Solución Codex
np.random.seed(123)
medias_bootstrap = []

for _ in range(1000):
    muestra = np.random.choice(ventas, size=len(ventas), replace=True)
    medias_bootstrap.append(muestra.mean())

ic_bootstrap = np.percentile(medias_bootstrap, [2.5, 97.5])
ic_bootstrap



#### Nota Codex

Bootstrap estima incertidumbre creando muchas muestras artificiales a partir de la muestra original. Es muy útil cuando queremos una aproximación práctica sin depender tanto de fórmulas teóricas.


## Ejercicio 10: error tipo I y tipo II


### Ejercicio a analizar

Responde en una frase: en un test A/B, ¿qué sería un error de tipo I y qué sería un error de tipo II?


In [ ]:

# Solución Codex
# Error de tipo I: concluir que B mejora la conversión cuando en realidad no mejora.
# Error de tipo II: no detectar que B mejora la conversión cuando en realidad sí mejora.



#### Nota Codex

El error de tipo I es una falsa alarma. El error de tipo II es dejar pasar un efecto real.


## Ejercicio 11: tamaño del efecto


### Ejercicio a analizar

Crea dos grupos de demanda: almacén A y almacén S. Calcula la diferencia de medias y el tamaño del efecto con Cohen's d.


In [ ]:

# Solución Codex
grupo_a = df.loc[df.Warehouse == 'Whse_A', 'Order_Demand']
grupo_s = df.loc[df.Warehouse == 'Whse_S', 'Order_Demand']

diferencia_medias = grupo_s.mean() - grupo_a.mean()
desv_pooled = np.sqrt((grupo_a.var(ddof=1) + grupo_s.var(ddof=1)) / 2)
cohens_d = diferencia_medias / desv_pooled

print('Media A:', grupo_a.mean())
print('Media S:', grupo_s.mean())
print('Diferencia de medias:', diferencia_medias)
print("Cohen's d:", cohens_d)



#### Nota Codex

El p-valor responde si hay evidencia estadística. El tamaño del efecto ayuda a responder si la diferencia es grande o pequeña en la práctica.


## Ejercicio 12: test no paramétrico


### Ejercicio a analizar

Compara `Order_Demand` entre los almacenes A y S usando Mann-Whitney. ¿Por qué podría tener sentido usar un test no paramétrico aquí?


In [ ]:

# Solución Codex
resultado_mann = stats.mannwhitneyu(grupo_a, grupo_s, alternative='two-sided')
resultado_mann



#### Nota Codex

Mann-Whitney puede ser útil cuando la distribución es muy asimétrica o hay muchos outliers y no queremos depender tanto de supuestos paramétricos.


## Ejercicio 13: ANOVA


### Ejercicio a analizar

Compara la media de `Order_Demand` entre tres almacenes usando ANOVA de una vía.


In [ ]:

# Solución Codex
almacenes = df.Warehouse.value_counts().index[:3]
grupos = [df.loc[df.Warehouse == almacen, 'Order_Demand'] for almacen in almacenes]

resultado_anova = stats.f_oneway(*grupos)
print('Almacenes comparados:', list(almacenes))
resultado_anova



#### Nota Codex

ANOVA indica si al menos una media de grupo parece diferente. Si el resultado es significativo, luego hacen falta comparaciones post-hoc para saber entre qué grupos está la diferencia.


## Ejercicio 14: normalidad y varianzas


### Ejercicio a analizar

Toma una muestra de 500 valores de `Order_Demand`. Aplica Shapiro-Wilk para normalidad y Levene para comparar varianzas entre almacenes A y S.


In [ ]:

# Solución Codex
muestra_ventas = ventas.sample(500, random_state=123)
shapiro = stats.shapiro(muestra_ventas)
levene = stats.levene(grupo_a.sample(500, random_state=123), grupo_s.sample(500, random_state=123))

print('Shapiro:', shapiro)
print('Levene:', levene)



#### Nota Codex

Con muestras grandes, los tests de normalidad pueden ser muy sensibles. Combina tests con gráficos y sentido práctico.


## Ejercicio 15: regresión simple y residuos


### Ejercicio a analizar

Crea una variable `indice` con la posición de cada registro en una muestra de 500 ventas. Ajusta una regresión lineal simple entre `indice` y `Order_Demand`, calcula predicciones y residuos.


In [ ]:

# Solución Codex
muestra = df.Order_Demand.sample(500, random_state=42).reset_index(drop=True)
indice = np.arange(len(muestra))

modelo = stats.linregress(indice, muestra)
predicciones = modelo.intercept + modelo.slope * indice
residuos = muestra - predicciones

print(modelo)
print('Media de residuos:', residuos.mean())
print('Desviación de residuos:', residuos.std())



#### Nota Codex

Los residuos son lo que el modelo no explica. Si muestran patrones fuertes, el modelo probablemente no captura bien la estructura de los datos.


## Ejercicio 16: corrección de Bonferroni


### Ejercicio a analizar

Imagina que haces 12 tests con `alpha = 0.05`. Calcula el nuevo alpha con corrección de Bonferroni. Luego decide si un p-valor de `0.01` seguiría siendo significativo.


In [ ]:

# Solución Codex
alpha = 0.05
numero_tests = 12
alpha_bonferroni = alpha / numero_tests
pvalor = 0.01

print('Alpha corregido:', alpha_bonferroni)
print('Significativo tras Bonferroni:', pvalor < alpha_bonferroni)



#### Nota Codex

Bonferroni reduce el riesgo de falsos positivos cuando haces muchos tests, pero también puede ser muy conservador y dificultar detectar efectos reales.


## Ejercicio 17: VIF y multicolinealidad


### Ejercicio a analizar

Crea tres variables numéricas artificiales: `x1`, `x2` casi igual a `x1`, y `x3` independiente. Calcula el VIF de cada una para detectar multicolinealidad.


In [ ]:

# Solución Codex
np.random.seed(123)
x1 = np.random.normal(size=200)
x2 = x1 * 0.95 + np.random.normal(scale=0.05, size=200)
x3 = np.random.normal(size=200)

X = pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3})

if variance_inflation_factor is None:
    print('statsmodels no está disponible en este entorno.')
else:
    vif = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })
    display(vif)



#### Nota Codex

VIF alto indica que una variable se puede explicar bastante bien a partir de las otras. Eso no siempre perjudica la predicción, pero sí complica la interpretación de coeficientes.



## Cierre

#### Nota Codex

Estos ejercicios no sustituyen el curso oficial: lo complementan. La clave es que aprendas a decidir qué herramienta usar según el tipo de variable, la pregunta estadística y las condiciones de los datos.
